<a href="https://colab.research.google.com/github/PrarthanaShende/AI-Projects-/blob/main/Poetry_Generation1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Poetry Generation**

Text Generation Model

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Data Gathering

In [2]:
with open("/content/Lil_Wayne.txt", "r", encoding="utf-8") as f:
    Lil_Wayne = f.read()

# Preview first 50 word
print(Lil_Wayne[:50])


﻿They call me Mr Carter I kissed the daughter
Of t


## Tokenization

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([Lil_Wayne])

total_words =len(tokenizer.word_index) + 1
print('vocabulary_size:',total_words)

index_word = {i: word for word, i in tokenizer.word_index.items()}

vocabulary_size: 3584


In [4]:
token_list =tokenizer.texts_to_sequences([Lil_Wayne])[0]

input_sequences = []

for i in range(5,len(token_list)):
  n_gram_sequence = token_list[i-5:i+1]   # 6‑word window (5 previous + current)
  input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[1480, 103, 12, 399, 233, 1], [103, 12, 399, 233, 1, 1481], [12, 399, 233, 1, 1481, 2], [399, 233, 1, 1481, 2, 533], [233, 1, 1481, 2, 533, 20]]


## PADDING

In [5]:
max_sequence_length = 6

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index

# Convert target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (23863, 5)
y shape: (23863, 3584)


## BUILD LSTM MODEL

In [6]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 128)         │       458,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3584)           │       361,984 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,098,636 (4.19 MB)

 Trainable params: 1,098,636 (4.19 MB)

 Non-trainable params: 0 (0.00 B)

## Model Training

In [7]:
es = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)
history = model.fit(X,y,epochs=40,batch_size=32,verbose=1,callbacks=[es])


Epoch 1/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.0383 - loss: 6.6320
Epoch 2/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0411 - loss: 6.2175
Epoch 3/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.0481 - loss: 5.9608
Epoch 4/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.0652 - loss: 5.6723
Epoch 5/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.0855 - loss: 5.4079
Epoch 6/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.1046 - loss: 5.1574
Epoch 7/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.1189 - loss: 4.9459
Epoch 8/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.1356 - loss: 4.7553
Epoch 9/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.1486 - loss: 4.5734
Epoch 10/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.1638 - loss: 4.3924
Epoch 11/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.1809 - loss: 4.2273
Epoch 12/40
746/746 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/st

In [8]:
model.save("Poetry Generation.keras")

In [12]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")


    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]

    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

In [13]:
def generate_text(seed_text, next_words, model, max_sequence_len, temperature=0.8, top_k=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        preds = model.predict(token_list, verbose=0)[0]
        predicted_index = sample_with_temperature(preds, temperature=temperature, top_k=top_k)

        seed_text += " " + index_word[predicted_index]
    return seed_text

## Prediction
print(generate_text("Once upon a dream", 20, model, max_sequence_length, temperature=0.7, top_k=5))


Once upon a dream and i tell the grip i pull a cloud and all i just wanna have to knock my couple walls
